# Model 4c: + Age and Minutes — Midfielders## Agregamos controlesAl modelo 4b le sumamos dos variables de control:$$\Delta PQ_i = \alpha + \underbrace{\sum_{j=1}^{17} \beta_j \cdot \text{from}_{Q_j}}_{\text{qualities previas}} + \underbrace{\sum_{k=1}^{7} \gamma_k \cdot \Delta TQ_k}_{\text{cambio táctico}} + \underbrace{\delta \cdot \text{age}}_{\text{edad}} + \underbrace{\epsilon \cdot \Delta \text{minutes}}_{\text{cambio en minutos}}$$¿Por qué?- **Age:** jugadores más jóvenes tienden a mejorar, jugadores mayores a empeorar- **Delta minutes:** si un jugador juega muchos más minutos en el nuevo equipo, puede que sus stats cambien simplemente por tener más tiempo en cancha (o al revés, si juega menos)26 features (17 player + 7 delta team + age + delta minutes).---

In [ ]:
import pandas as pdimport numpy as npimport statsmodels.api as smfrom sklearn.metrics import r2_score, mean_absolute_error, mean_squared_errorimport plotly.graph_objects as goimport plotly.express as pxfrom plotly.subplots import make_subplotsimport warningswarnings.filterwarnings("ignore")pd.set_option("display.max_columns", 40)pd.set_option("display.float_format", "{:.4f}".format)DATA = "../../../../thesis_data/processed_data/thesis_model_dataset/active/within_league_transfers_v5.parquet"df = pd.read_parquet(DATA)mf = df[df["from_position"] == "Midfielder"].copy()print(f"Midfielders: {len(mf):,} rows")

In [ ]:
QUALITIES = [    "Active defence", "Aerial threat", "Box threat", "Composure",    "Defensive heading", "Dribbling", "Effectiveness", "Finishing",    "Hold-up play", "Intelligent defence", "Involvement",    "Passing quality", "Pressing", "Progression",    "Providing teammates", "Run quality", "Winning duels",]TEAM_Q = ["DEFENCE", "DEFENSIVE_TRANSITION", "ATTACKING_TRANSITION",          "ATTACK", "PENETRATION", "CHANCE_CREATION", "OUTCOME"]TEAM_Q_LABELS = {    "DEFENCE": "Defence", "DEFENSIVE_TRANSITION": "Def. Transition",    "ATTACKING_TRANSITION": "Att. Transition", "ATTACK": "Attack",    "PENETRATION": "Penetration", "CHANCE_CREATION": "Chance Creation",    "OUTCOME": "Outcome",}from_pq = [f"from_{q}" for q in QUALITIES]to_pq   = [f"to_{q}" for q in QUALITIES]from_tq = [f"from_q_proj_{q}" for q in TEAM_Q]to_tq   = [f"to_q_{q}" for q in TEAM_Q]# Compute deltasfor q in QUALITIES:    mf[f"delta_{q}"] = mf[f"to_{q}"] - mf[f"from_{q}"]for q in TEAM_Q:    mf[f"delta_team_{q}"] = mf[f"to_q_{q}"] - mf[f"from_q_proj_{q}"]mf["delta_Minutes"] = mf["to_Minutes"] - mf["from_Minutes"]delta_pq = [f"delta_{q}" for q in QUALITIES]delta_tq = [f"delta_team_{q}" for q in TEAM_Q]

## Distribución de los Controles

In [ ]:
fig = make_subplots(rows=1, cols=2, subplot_titles=["Age distribution", "Delta Minutes distribution"])fig.add_trace(go.Histogram(x=mf["player_season_age"].dropna(), nbinsx=30, marker_color="#636EFA"), row=1, col=1)fig.add_trace(go.Histogram(x=mf["delta_Minutes"].dropna(), nbinsx=50, marker_color="#EF553B"), row=1, col=2)fig.update_xaxes(title_text="Age", row=1, col=1)fig.update_xaxes(title_text="Delta Minutes", row=1, col=2)fig.update_layout(height=350, template="plotly_white", showlegend=False,    title="Distribución de variables de control")fig.show()

---## Resultados17 pre-transfer qualities + 7 delta team qualities + age + delta minutes → predice cada `delta_Qi` por separado (17 modelos OLS).

In [ ]:
features = from_pq + delta_tq + ['player_season_age', 'delta_Minutes']# Clean + splitkeep = features + delta_pq + ["from_season"]clean = mf[keep].dropna()train = clean[clean["from_season"] <= 2023]test  = clean[clean["from_season"] == 2024]print(f"Clean: {len(clean):,} | Train: {len(train):,} | Test: {len(test):,}")print(f"Features: {len(features)}")results = []models = {}for q in QUALITIES:    target = f"delta_{q}"    X_train = sm.add_constant(train[features])    X_test  = sm.add_constant(test[features])    y_train = train[target]    y_test  = test[target]    model = sm.OLS(y_train, X_train).fit()    y_pred = model.predict(X_test)    results.append({        "Quality": q,        "R²_train": model.rsquared,        "R²_adj": model.rsquared_adj,        "R²_test": r2_score(y_test, y_pred),        "MAE_test": mean_absolute_error(y_test, y_pred),        "F_pvalue": model.f_pvalue,    })    models[q] = modelres = pd.DataFrame(results)print(res.to_string(index=False))print(f"\nMean R²_train: {res['R²_train'].mean():.4f}")print(f"Mean R²_adj:   {res['R²_adj'].mean():.4f}")print(f"Mean R²_test:  {res['R²_test'].mean():.4f}")print(f"Mean MAE_test: {res['MAE_test'].mean():.4f}")

### Comparación con modelos anteriores

In [ ]:
# Recompute M3, M4a, M4b for comparisonm3_features = from_pq + from_tq + to_tqm3_keep = m3_features + to_pq + ["from_season"]m3_clean = mf[m3_keep].dropna()m3_train = m3_clean[m3_clean["from_season"] <= 2023]m3_test  = m3_clean[m3_clean["from_season"] == 2024]m3_r2 = []for q in QUALITIES:    X_tr = sm.add_constant(m3_train[m3_features])    X_te = sm.add_constant(m3_test[m3_features])    m = sm.OLS(m3_train[f"to_{q}"], X_tr).fit()    m3_r2.append(r2_score(m3_test[f"to_{q}"], m.predict(X_te)))m4a_feat = delta_tqm4a_keep = m4a_feat + delta_pq + ["from_season"]m4a_clean = mf[m4a_keep].dropna()m4a_train = m4a_clean[m4a_clean["from_season"] <= 2023]m4a_test  = m4a_clean[m4a_clean["from_season"] == 2024]m4a_r2 = []for q in QUALITIES:    X_tr = sm.add_constant(m4a_train[m4a_feat])    X_te = sm.add_constant(m4a_test[m4a_feat])    m = sm.OLS(m4a_train[f"delta_{q}"], X_tr).fit()    m4a_r2.append(r2_score(m4a_test[f"delta_{q}"], m.predict(X_te)))m4b_feat = from_pq + delta_tqm4b_keep = m4b_feat + delta_pq + ["from_season"]m4b_clean = mf[m4b_keep].dropna()m4b_train = m4b_clean[m4b_clean["from_season"] <= 2023]m4b_test  = m4b_clean[m4b_clean["from_season"] == 2024]m4b_r2 = []for q in QUALITIES:    X_tr = sm.add_constant(m4b_train[m4b_feat])    X_te = sm.add_constant(m4b_test[m4b_feat])    m = sm.OLS(m4b_train[f"delta_{q}"], X_tr).fit()    m4b_r2.append(r2_score(m4b_test[f"delta_{q}"], m.predict(X_te)))comp = pd.DataFrame({"Quality": QUALITIES, "R²_M3_levels": m3_r2, "R²_M4a_delta_tq": m4a_r2, "R²_M4b_pq+dtq": m4b_r2})comp["R²_M4c_+controls"] = res["R²_test"].valuesprint(comp.to_string(index=False))print(f"\nMean R²_test per model:")for col in [c for c in comp.columns if c.startswith("R²")]:    print(f"  {col}: {comp[col].mean():.4f}")

In [ ]:
# Bar chart comparisonfig = go.Figure()r2_cols = [c for c in comp.columns if c.startswith("R²")]colors = ["#636EFA", "#EF553B", "#00CC96", "#AB63FA", "#FFA15A", "#19D3F3"]for i, col in enumerate(r2_cols):    fig.add_trace(go.Bar(name=col.replace("R²_", ""), x=comp["Quality"], y=comp[col],        marker_color=colors[i % len(colors)]))fig.update_layout(title="R²_test — Comparación de modelos (Midfielders)",    yaxis_title="R²_test", barmode="group", height=450, template="plotly_white",    xaxis_tickangle=-45)fig.show()

### Full OLS Summary — Best and Worst

In [ ]:
best_q = res.loc[res["R²_test"].idxmax(), "Quality"]worst_q = res.loc[res["R²_test"].idxmin(), "Quality"]print(f"BEST: {best_q} (R²_test = {res.loc[res['R²_test'].idxmax(), 'R²_test']:.4f})")print(models[best_q].summary())print(f"\n{'='*70}")print(f"WORST: {worst_q} (R²_test = {res.loc[res['R²_test'].idxmin(), 'R²_test']:.4f})")print(models[worst_q].summary())

### ¿Importan age y delta minutes?Coeficientes de age y delta_Minutes across all 17 models.

In [ ]:
control_coefs = []for q in QUALITIES:    m = models[q]    control_coefs.append({        "Quality": q,        "age_coef": m.params["player_season_age"],        "age_pvalue": m.pvalues["player_season_age"],        "dmin_coef": m.params["delta_Minutes"],        "dmin_pvalue": m.pvalues["delta_Minutes"],    })cc = pd.DataFrame(control_coefs)print(cc.to_string(index=False))print(f"\nAge significant (p<0.05): {(cc['age_pvalue'] < 0.05).sum()} / {len(cc)}")print(f"Delta Minutes significant (p<0.05): {(cc['dmin_pvalue'] < 0.05).sum()} / {len(cc)}")

---## TakeawayAgregar age y delta minutes como controles muestra:- **Age:** efecto generalmente pequeño. Si es significativo, confirma que jugadores más jóvenes tienden a mejorar.- **Delta minutes:** puede ser significativo para algunas qualities — más minutos jugados pueden correlacionar con cambios en performance.La mejora en R²_test vs modelo 4b nos dice si estas variables de control aportan poder predictivo real o no.**Siguiente paso:** Regularización (Lasso/Ridge) sobre la mejor especificación para evitar overfitting y hacer feature selection automática.